In [ ]:
import nibabel as nib
import numpy as np
# Extract BP from all_labels_mask.nii
# 1+2=LVBP, 4=RVBP, 22=TCV

for case in [1]: 
    path=f"/data5/alee9/MS_HF_cohort/data/{case:02d}/RV/"
    maskfile=path+"/all_labels_mask_downsample.nii.gz" # mask file with all labels
    mask_img = nib.load(maskfile)
    mask_img.set_sform(mask_img.get_qform(), code=1)
    # Define target labels
    target_labels = [1,2,4,22]
    mask_data = mask_img.get_fdata()

    # Create binary mask: 1 where label is in target list, else 0
    binary_mask = np.isin(mask_data, target_labels).astype(np.uint8)
    print('create binary mask from labels')

    # Apply mask (binary mask: keep where mask > 0)   
    masked_data = np.where(binary_mask > 0, mask_data, 0)
    masked_data = np.where(masked_data==2, 1, masked_data)
    masked_data = np.where(masked_data==22, 3, masked_data)
    masked_data = np.uint8(masked_data*10) # LVBP=10, RVVP=40, TCV=30

    

    # Create new image with original affine and header
    masked_img = nib.Nifti1Image(masked_data, affine=mask_img.affine, header=mask_img.header)

    # Save result
    nib.save(masked_img, path+'BloodPools.nii')

FileNotFoundError: No such file or no access: '/data5/alee9/MS_HF_cohort/data/07/RV/all_labels_mask_downsample.nii'

In [ ]:
import nibabel as nib
from pathlib import Path
import numpy as np

for case in [1]: # range(1,24): #range(20,21): # range(1, 24): 
    print(f"############## Processing case {case:02d} ##############")
    path=f"/data5/alee9/MS_HF_cohort/data/{case:02d}/RV/"
    print(path)
    maskfile=path+"/all_labels_mask.nii" # mask file with all labels
    print(maskfile)
    if Path(maskfile).exists(): 
        print("Mask file exists")
    else:
        print("Mask file does not exist")
        continue
    # Load the dcm image and the mask
    target_labels= [3, 5] #, 21, 22, 23, 24] # 2=LVpap 

    # Remove non-target labels
    mask_img = nib.load(maskfile)
    mask_img.set_sform(mask_img.get_qform(), code=1)
    mask_data = mask_img.get_fdata()
    mask_data = mask_data.astype(np.int8)

    # Create binary mask: 1 where label is in target list, else 0
    binary_mask = np.isin(mask_data, target_labels).astype(np.uint8)
    
    mask_data = binary_mask * mask_data  # Apply binary mask to original data
    # Convert to int8
    #mask_img.set_fdata(mask_data)  # Update the mask image with the new data
    masked_img = nib.Nifti1Image(mask_data, affine=mask_img.affine, header=mask_img.header)  
    nib.save(masked_img, path+'BiV_masked.nii')
    print('Case %02d done' % case)
    print(path+'BiV_masked.nii')

############## Processing case 07 ##############
/data5/alee9/MS_HF_cohort/data/07/RV/
/data5/alee9/MS_HF_cohort/data/07/RV//all_labels_mask_downsample.nii
Mask file exists
Case 07 done
/data5/alee9/MS_HF_cohort/data/07/RV/BiV_masked.nii


In [ ]:
import pyvista as pv
import numpy as np  



filename='/home/alee9/Work/MS_HF_cohort//data/01/1_seg/final_network_flip.vtk'
mesh = pv.read(filename)
HB_indices = np.where(mesh.cell_data['elemTag'] == 150)[0]
LB_indices = np.where(mesh.cell_data['elemTag'] == 151)[0]
RB_indices = np.where(mesh.cell_data['elemTag'] == 152)[0]
LV_fascicles_indices = np.where(np.isin(mesh.cell_data['elemTag'], [153, 154, 155]))[0]
RV_fascicles_indices = np.where(np.isin(mesh.cell_data['elemTag'], [156, 157]))[0]

HB=mesh.extract_cells(HB_indices).points
LB=mesh.extract_cells(LB_indices).points
RB=mesh.extract_cells(RB_indices).points
LV_fascicles=mesh.extract_cells(LV_fascicles_indices).points
RV_fascicles=mesh.extract_cells(RV_fascicles_indices).points



[[ -10.7088 -104.464   381.811 ]
 [ -11.2411 -104.069   381.31  ]
 [ -11.8348 -103.597   380.761 ]
 [ -12.5988 -103.094   380.187 ]
 [ -13.3825 -102.592   379.838 ]
 [ -14.2429 -101.964   379.732 ]
 [ -15.0298 -101.428   379.621 ]
 [ -11.8348 -103.597   380.761 ]
 [ -12.5988 -103.094   380.187 ]
 [ -13.3825 -102.592   379.838 ]
 [ -14.2429 -101.964   379.732 ]
 [ -15.0298 -101.428   379.621 ]]


In [5]:
# Check RV BP 
import nibabel as nib
from pathlib import Path
import numpy as np

for case in  [1, 2, 4, 6, 7, 8, 9, 10, 11, 13]: # range(1,24): #range(20,21): # range(1, 24): 
    print(f"############## Processing case {case:02d} ##############")
    path=f"/home/alee9/Work/MS_HF_cohort/data/{case:02d}/1_seg/"
    print(path)
    maskfile=path+"/all_labels_mask.nii" # mask file with all labels
    print(maskfile)
    if Path(maskfile).exists(): 
        print("Mask file exists")
    else:
        print("Mask file does not exist")
        continue
    # Load the dcm image and the mask
    target_labels= [4] # 4 = RVBP

    # Remove non-target labels
    mask_img = nib.load(maskfile)
    mask_img.set_sform(mask_img.get_qform(), code=1)
    mask_data = mask_img.get_fdata()
    mask_data = mask_data.astype(np.int8)
    
    # Calculate RVBP volume
    # Count voxels with label 4 (RVBP)
    rvbp_voxels = np.sum(mask_data == 4)
    #print(f"Number of RVBP voxels: {rvbp_voxels}")
    
    # Get voxel dimensions from affine matrix
    affine = mask_img.affine
    voxel_dims = np.abs(np.diag(affine)[:3])  # Get x, y, z dimensions
    voxel_volume = np.prod(voxel_dims)  # Volume of single voxel in mm³
    
    #print(f"Voxel dimensions (mm): {voxel_dims}")
    #print(f"Single voxel volume (mm³): {voxel_volume:.4f}")
    
    # Calculate total RVBP volume
    rvbp_volume_mm3 = rvbp_voxels * voxel_volume
    rvbp_volume_ml = rvbp_volume_mm3 / 1000  # Convert mm³ to mL
    
    #print(f"RVBP volume: {rvbp_volume_mm3:.2f} mm³")
    print(f"RVBP volume: {rvbp_volume_ml:.2f} mL")
    #print(f"RVBP volume: {rvbp_volume_ml/1000:.4f} L")
    print("="*50)

    # Create binary mask: 1 where label is in target list, else 0
    #binary_mask = np.isin(mask_data, target_labels).astype(np.uint8)

############## Processing case 01 ##############
/home/alee9/Work/MS_HF_cohort/data/01/1_seg/
/home/alee9/Work/MS_HF_cohort/data/01/1_seg//all_labels_mask.nii
Mask file exists
RVBP volume: 233.51 mL
############## Processing case 02 ##############
/home/alee9/Work/MS_HF_cohort/data/02/1_seg/
/home/alee9/Work/MS_HF_cohort/data/02/1_seg//all_labels_mask.nii
Mask file exists
RVBP volume: 273.21 mL
############## Processing case 04 ##############
/home/alee9/Work/MS_HF_cohort/data/04/1_seg/
/home/alee9/Work/MS_HF_cohort/data/04/1_seg//all_labels_mask.nii
Mask file exists
RVBP volume: 236.50 mL
############## Processing case 06 ##############
/home/alee9/Work/MS_HF_cohort/data/06/1_seg/
/home/alee9/Work/MS_HF_cohort/data/06/1_seg//all_labels_mask.nii
Mask file exists
RVBP volume: 214.96 mL
############## Processing case 07 ##############
/home/alee9/Work/MS_HF_cohort/data/07/1_seg/
/home/alee9/Work/MS_HF_cohort/data/07/1_seg//all_labels_mask.nii
Mask file exists
RVBP volume: 230.82 mL
#####